# County Nutrient Loading — As-of Match + Partial Manure Refresh (2015–2025)

Companion to **P3**. P3 (`county-agriculture.csv`) is the faithful raw
merge across all *native* years; the USGS/Falcone N&P fertilizer & manure series
are quinquennial and **end in 2017**, so within the 2015–2025 water-quality window
only **2017** has a native value. This notebook produces a **modeling-ready**
county × year nutrient table, dense for **every year 2015–2025**, so the S2
secondary merge can join it to station-years on exact `county_fips` + `year`.

## Method

**As-of matching (backward).** Each target year `Y` is anchored to the most recent
Falcone census year `≤ min(Y, 2017)` — its `np_base_year`. So 2015–2016 → 2012,
2017 → 2017, and 2018–2025 carry 2017 forward. Backward (`≤`) matching avoids
using future census data to represent an earlier year.

**Fertilizer N&P** → pure as-of / carry-forward (no recent county-level driver
exists to refresh it against; statewide fertilizer is comparatively flat).

**Manure N&P** → as-of baseline, then a **partial refresh** that scales the 2017
baseline by observed head-count change, holding Falcone's 2017 per-head nutrient
rate fixed (`manure_2017 × head_Y / head_2017`):

| Category | Refresh | Source |
|---|---|---|
| **Cattle** | annual, 2018–2025 | NASS `CATTLE, INCL CALVES` SURVEY inventory (99 counties, every year) |
| **Hogs** | 2022 census step only, applied to 2022–2025 | NASS `HOGS` CENSUS inventory (2017 & 2022; annual county hog counts aren't published) |
| **Poultry / Other** | none — carried from 2017 | not available at county resolution |

**Why not use `manure-weight-coefficients`?** That table holds *live-weight*
factors, not kg-N/kg-P excretion per head, so it can't turn head counts into
nutrient mass directly. Ratio-scaling against Falcone's own 2017 figures sidesteps
the need for absolute excretion coefficients and stays county-specific.

**Manure `Total` is recomputed** as Cattle+Hogs+Poultry+Other after refresh, so it
stays consistent with its (now partly updated) components.

## Provenance columns
`np_base_year`, `np_years_stale`, `manure_cattle_refreshed`,
`manure_hogs_refreshed`, `cattle_head_ratio`, `hog_head_ratio` — so no value ever
pretends to be observed when it was carried or scaled.

## Known limitations
- **No annual hog/CAFO trend.** County hog inventory is census-only (2017, 2022);
  2018–2021 carry 2017, 2023–2025 carry the 2022 ratio. A single step, not a curve.
- **Poultry & minor species are frozen at 2017.**
- 2015–2016 use the 2012 base with **no** refresh (pre-dates the NASS ratio base).

**Output:** `data/03a_merge_primary/county-agriculture-asof.csv`, one row per
`county_fips` × `year` (99 counties × 11 years = 1,089 rows).

In [1]:
import os
import numpy as np
import pandas as pd

AG = "../../data/tabular/02_clean/agriculture"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/county-agriculture-asof.csv"

TARGET_YEARS = list(range(2015, 2026))          # water-quality window
FALCONE_YEARS = [1950, 1954, 1959, 1964, 1969, 1974, 1978, 1982,
                 1987, 1992, 1997, 2002, 2007, 2012, 2017]
CAP_YEAR = 2017                                  # last native Falcone year
HOG_CENSUS_YEARS = (2017, 2022)


def base_year(y):
    """Most recent Falcone census year <= min(y, CAP_YEAR) (backward as-of)."""
    return max(yr for yr in FALCONE_YEARS if yr <= min(y, CAP_YEAR))


def std_fips(df):
    df = df.copy()
    df["county_fips"] = pd.to_numeric(df["county_fips"], errors="coerce")
    df = df.dropna(subset=["county_fips"])
    df["county_fips"] = df["county_fips"].astype(int).astype(str).str.zfill(5)
    return df


print("Base-year map:", {y: base_year(y) for y in TARGET_YEARS})

Base-year map: {2015: 2012, 2016: 2012, 2017: 2017, 2018: 2017, 2019: 2017, 2020: 2017, 2021: 2017, 2022: 2017, 2023: 2017, 2024: 2017, 2025: 2017}


## Step 1: Head-count ratios (cattle annual, hog census)

Cattle: `CATTLE, INCL CALVES` INVENTORY, SURVEY program — a fully-populated annual
county series. Ratio = `head_Y / head_2017`. Hogs: `HOGS` INVENTORY, CENSUS
program — ratio = `head_2022 / head_2017`, applied to 2022–2025. Counties missing a
count fall back to ratio = 1 (carry-forward), flagged as not-refreshed.

In [2]:
lv = std_fips(pd.read_csv(f"{AG}/livestock-inventory-clean.csv"))
lv = lv[lv["domain"] == "TOTAL"]

# --- cattle: annual SURVEY inventory ---
cat = lv[(lv["commodity_detail"] == "CATTLE, INCL CALVES")
         & (lv["statistic"] == "INVENTORY")
         & (lv["program"] == "SURVEY")]
cat_wide = cat.pivot_table(index="county_fips", columns="year", values="value", aggfunc="first")
cattle_base = cat_wide[CAP_YEAR]

cattle_ratio = pd.DataFrame(index=cat_wide.index)
for y in TARGET_YEARS:
    if y <= CAP_YEAR:
        cattle_ratio[y] = 1.0                    # native / no refresh
    else:
        cattle_ratio[y] = cat_wide.get(y) / cattle_base   # NaN where either missing

# --- hogs: 2017 & 2022 census inventory ---
hog = lv[(lv["commodity_detail"] == "HOGS")
         & (lv["statistic"] == "INVENTORY")
         & (lv["program"] == "CENSUS")]
hog_wide = hog.pivot_table(index="county_fips", columns="year", values="value", aggfunc="first")
hog_ratio_2022 = hog_wide[HOG_CENSUS_YEARS[1]] / hog_wide[HOG_CENSUS_YEARS[0]]

hog_ratio = pd.DataFrame(index=hog_wide.index)
for y in TARGET_YEARS:
    hog_ratio[y] = hog_ratio_2022 if y >= HOG_CENSUS_YEARS[1] else 1.0

print(f"cattle ratio base counties: {cattle_base.notna().sum()}")
print(f"hog ratio (2022/2017) counties: {hog_ratio_2022.notna().sum()}")
print("cattle ratio 2025 describe:\n", cattle_ratio[2025].describe().round(3).to_string())

cattle ratio base counties: 99
hog ratio (2022/2017) counties: 94
cattle ratio 2025 describe:
 count    99.000
mean      0.880
std       0.206
min       0.400
25%       0.748
50%       0.873
75%       0.971
max       1.660


## Step 2: Fertilizer N&P — as-of / carry-forward

For each target year, take the fertilizer rows at that year's `np_base_year` and
relabel them to the target year. No refresh.

In [3]:
fert = std_fips(pd.read_csv(f"{AG}/np-fertilizer-clean.csv"))

# The Falcone `total` source is null for the 2012/2017 base years, so keep the
# two real application contexts (farm/nonfarm) and recompute total = farm+nonfarm.
fert = fert[fert["source"].isin(["farm", "nonfarm"])]

fert_parts = []
for y in TARGET_YEARS:
    b = base_year(y)
    part = fert[fert["year"] == b][["county_fips", "nutrient", "source", "value_kg"]].copy()
    part["year"] = y
    fert_parts.append(part)
fert_asof = pd.concat(fert_parts, ignore_index=True)

fert_wide = fert_asof.pivot_table(
    index=["county_fips", "year"], columns=["nutrient", "source"], values="value_kg", aggfunc="first")
fert_wide.columns = [f"npfert__{n.lower()}__{s}_kg" for n, s in fert_wide.columns]
fert_wide = fert_wide.reset_index()
for nut in ["n", "p"]:
    fert_wide[f"npfert__{nut}__total_kg"] = fert_wide[
        [f"npfert__{nut}__farm_kg", f"npfert__{nut}__nonfarm_kg"]].sum(axis=1, min_count=1)
print(f"fertilizer: {fert_wide.shape[0]:,} county-years x {fert_wide.shape[1]-2} cols")
fert_wide.head(3)

fertilizer: 1,089 county-years x 6 cols


,county_fips,year,npfert__n__farm_kg,npfert__n__nonfarm_kg,npfert__p__farm_kg,npfert__p__nonfarm_kg,npfert__n__total_kg,npfert__p__total_kg
0,19001,2015,8.361537e+06,8492.000000,1.183782e+06,2900.000000,8.370029e+06,1.186682e+06
1,19001,2016,8.361537e+06,8492.000000,1.183782e+06,2900.000000,8.370029e+06,1.186682e+06
2,19001,2017,9.050257e+06,7387.836029,1.454374e+06,2292.867628,9.057645e+06,1.456667e+06


## Step 3: Manure N&P — as-of baseline + partial refresh

Carry each year's `np_base_year` manure baseline, multiply Cattle by its annual
head-ratio and Hogs by the 2022 census ratio (1.0 elsewhere), then recompute
`Total` from the four refreshed components.

In [4]:
man = std_fips(pd.read_csv(f"{AG}/np-manure-clean.csv"))
components = ["Cattle", "Hogs", "Poultry", "Other"]

man_parts = []
for y in TARGET_YEARS:
    b = base_year(y)
    part = man[(man["year"] == b) & (man["animal_category"].isin(components))][
        ["county_fips", "animal_category", "nutrient", "value_kg"]].copy()
    part["year"] = y
    man_parts.append(part)
man_asof = pd.concat(man_parts, ignore_index=True)

# per (county, year) multipliers, with carry-forward fallback (ratio -> 1)
cat_r = cattle_ratio.reset_index().melt(id_vars="county_fips", var_name="year", value_name="cattle_head_ratio")
hog_r = hog_ratio.reset_index().melt(id_vars="county_fips", var_name="year", value_name="hog_head_ratio")
man_asof = man_asof.merge(cat_r, on=["county_fips", "year"], how="left") \
                   .merge(hog_r, on=["county_fips", "year"], how="left")

mult = np.ones(len(man_asof))
is_cattle = man_asof["animal_category"] == "Cattle"
is_hog = man_asof["animal_category"] == "Hogs"
mult[is_cattle] = man_asof.loc[is_cattle, "cattle_head_ratio"].fillna(1.0)
mult[is_hog] = man_asof.loc[is_hog, "hog_head_ratio"].fillna(1.0)
man_asof["value_kg"] = man_asof["value_kg"] * mult

# recompute Total = sum of refreshed components
total = man_asof.groupby(["county_fips", "year", "nutrient"], as_index=False)["value_kg"].sum()
total["animal_category"] = "Total"
man_full = pd.concat([man_asof[["county_fips", "year", "animal_category", "nutrient", "value_kg"]],
                      total], ignore_index=True)

man_wide = man_full.pivot_table(
    index=["county_fips", "year"], columns=["animal_category", "nutrient"], values="value_kg", aggfunc="first")
man_wide.columns = [f"npmanure__{a.lower()}__{n.lower()}_kg" for a, n in man_wide.columns]
man_wide = man_wide.reset_index()
print(f"manure: {man_wide.shape[0]:,} county-years x {man_wide.shape[1]-2} cols")
man_wide.head(3)

manure: 1,089 county-years x 10 cols


,county_fips,year,npmanure__cattle__n_kg,npmanure__cattle__p_kg,npmanure__hogs__n_kg,npmanure__hogs__p_kg,npmanure__other__n_kg,npmanure__other__p_kg,npmanure__poultry__n_kg,npmanure__poultry__p_kg,npmanure__total__n_kg,npmanure__total__p_kg
0,19001,2015,1.912949e+06,581155.782959,389947.651897,173310.067510,32551.380227,5646.509605,964.036808,359.096020,2.336412e+06,760471.456093
1,19001,2016,1.912949e+06,581155.782959,389947.651897,173310.067510,32551.380227,5646.509605,964.036808,359.096020,2.336412e+06,760471.456093
2,19001,2017,2.187355e+06,650227.884845,579151.553878,257400.690613,30561.538485,5301.973650,911.379580,346.132583,2.797980e+06,913276.681690


## Step 4: Provenance table + assemble + save

In [5]:
prov_rows = []
for y in TARGET_YEARS:
    b = base_year(y)
    for cfips in cattle_ratio.index:
        cr = cattle_ratio.at[cfips, y] if cfips in cattle_ratio.index else np.nan
        hr = hog_ratio.at[cfips, y] if cfips in hog_ratio.index else np.nan
        prov_rows.append({
            "county_fips": cfips, "year": y,
            "np_base_year": b, "np_years_stale": y - b,
            "cattle_head_ratio": 1.0 if pd.isna(cr) else cr,
            "hog_head_ratio": 1.0 if pd.isna(hr) else hr,
            "manure_cattle_refreshed": bool(y > CAP_YEAR and not pd.isna(cr)),
            "manure_hogs_refreshed": bool(y >= HOG_CENSUS_YEARS[1] and not pd.isna(hr)),
        })
prov = pd.DataFrame(prov_rows)

df = prov.merge(fert_wide, on=["county_fips", "year"], how="left") \
         .merge(man_wide, on=["county_fips", "year"], how="left")
df = df.sort_values(["county_fips", "year"]).reset_index(drop=True)

assert not df.duplicated(subset=["county_fips", "year"]).any(), "grain violated"
assert len(df) == 99 * len(TARGET_YEARS), f"expected {99*len(TARGET_YEARS)} rows, got {len(df)}"

print(f"Final shape: {df.shape}")
print(f"County-years: {len(df):,} | counties: {df['county_fips'].nunique()} | years: {df['year'].min()}-{df['year'].max()}")
print("\nRefresh coverage (county-years):")
print(f"  cattle refreshed: {df['manure_cattle_refreshed'].sum():,}")
print(f"  hogs refreshed:   {df['manure_hogs_refreshed'].sum():,}")
print("\nStatewide manure N by year (kg) — carried vs refreshed effect:")
print(df.groupby('year')['npmanure__total__n_kg'].sum().round(0).to_string())
df.head(3)

Final shape: (1089, 24)
County-years: 1,089 | counties: 99 | years: 2015-2025

Refresh coverage (county-years):
  cattle refreshed: 792
  hogs refreshed:   376

Statewide manure N by year (kg) — carried vs refreshed effect:
year
2015    420285275.0
2016    420285275.0
2017    463284127.0
2018    469061646.0
2019    466670000.0
2020    461970027.0
2021    449952045.0
2022    468513934.0
2023    456507949.0
2024    454148700.0
2025    456507250.0


,county_fips,year,np_base_year,np_years_stale,cattle_head_ratio,hog_head_ratio,manure_cattle_refreshed,manure_hogs_refreshed,npfert__n__farm_kg,npfert__n__nonfarm_kg,...,npmanure__cattle__n_kg,npmanure__cattle__p_kg,npmanure__hogs__n_kg,npmanure__hogs__p_kg,npmanure__other__n_kg,npmanure__other__p_kg,npmanure__poultry__n_kg,npmanure__poultry__p_kg,npmanure__total__n_kg,npmanure__total__p_kg
0,19001,2015,2012,3,1.0,1.0,False,False,8.361537e+06,8492.000000,...,1.912949e+06,581155.782959,389947.651897,173310.067510,32551.380227,5646.509605,964.036808,359.096020,2.336412e+06,760471.456093
1,19001,2016,2012,4,1.0,1.0,False,False,8.361537e+06,8492.000000,...,1.912949e+06,581155.782959,389947.651897,173310.067510,32551.380227,5646.509605,964.036808,359.096020,2.336412e+06,760471.456093
2,19001,2017,2017,0,1.0,1.0,False,False,9.050257e+06,7387.836029,...,2.187355e+06,650227.884845,579151.553878,257400.690613,30561.538485,5301.973650,911.379580,346.132583,2.797980e+06,913276.681690


In [6]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 1,089 rows x 24 cols -> ../../data/03a_merge_primary/county-agriculture-asof.csv
